<a href="https://colab.research.google.com/github/Walid75364/GenIA/blob/Bootcamp_Weeks/W6_D4_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Daily Challenge

**Part 1: Load Documents & Execute Reranking Model**


In [1]:
pip install pinecone==6.0.1 pinecone-notebooks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 2.0 MB/s eta 0:00:00


In [6]:
import os
if not os.environ.get("PINECONE_API_KEY"):
   from pinecone_notebooks.colab import Authenticate
   Authenticate()

In [7]:
from pinecone import Pinecone
api_key = os.environ["PINECONE_API_KEY"]
environment = "us-east-1"  # e.g., "us-west1-gcp"
pc = Pinecone(api_key=api_key, environment=environment)

In [8]:
query = "Tell me about Apple's products"
documents = [
    "An apple a day keeps the doctor away, but Apple Inc. has revolutionized technology with its innovative products.",
    "I bought a fresh apple from the market yesterday, and Apple Inc. announced new features for the iPhone.",
    "The orchard was filled with ripe apples ready for harvest, just as Apple Inc. continues to grow its market share worldwide.",
    "She packed a crunchy apple for her lunch while reading about Apple Inc.'s latest earnings report.",
    "Apple is a popular fruit, but Apple Inc. is one of the most valuable companies in the world."
]

In [10]:
from pinecone import RerankModel
reranked = pc.inference.rerank(
   model="bge-reranker-v2-m3",
   query=query,
   documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
   top_n=3  # e.g., 3
)

In [11]:
def show_reranked(query, matches):
    print(f"Query: {query}")
    for i, m in enumerate(matches):
        print(f"{i+1}. Score: {m.score:.4f} | Document: {m.document.text}")

**Part 2: Setup a Serverless Index for Medical Notes**




In [12]:
pip install pandas torch transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 926.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 43.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [16]:
import os, time, pandas as pd, torch
from pinecone import Pinecone, ServerlessSpec

cloud = "aws"        # e.g., "aws"
region = "us-east-1"     # e.g., "us-east-1"
spec = ServerlessSpec(cloud="aws", region="us-east-1")
index_name = "pinecone-reranker"

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"], environment=f"{cloud}-{region}")

In [19]:
# Delete the index if it already exists
if pc.has_index(index_name):
    pc.delete_index(index_name)

# Create the index with the correct embedding dimension
pc.create_index(
    name=index_name,
    dimension=384,  # Must match the embedding model output size
    spec = ServerlessSpec(cloud="aws", region="us-east-1")
)


{
    "name": "pinecone-reranker",
    "metric": "cosine",
    "host": "pinecone-reranker-6afthmp.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

**Part 3: Load the Sample Data**

In [23]:
import os, requests, tempfile, pandas as pd

with tempfile.TemporaryDirectory() as tmpdir:
    file_path = os.path.join(tmpdir, "sample_notes_data.jsonl")

    # 🔧 Remplace l’URL par celle-ci :
    url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"

    resp = requests.get(url)
    resp.raise_for_status()
    open(file_path, "wb").write(resp.content)

    df = pd.read_json(file_path, orient='records', lines=True)
    print(df.head())



     id                                             values  \
0  P011  [-0.2027486265, 0.2769146562, -0.1509393603, 0...   
1  P001  [0.1842793673, 0.4459365904, -0.0770567134, 0....   
2  P002  [-0.2040648609, -0.1739618927, -0.2897160649, ...   
3  P003  [0.1889383644, 0.2924542725, -0.2335938066, -0...   
4  P004  [-0.12171068040000001, 0.1674752235, -0.231888...   

                                            metadata  
0  {'advice': 'rest, hydrate', 'symptoms': 'heada...  
1  {'tests': 'EKG, stress test', 'symptoms': 'che...  
2  {'HbA1c': '7.2', 'condition': 'diabetes', 'med...  
3  {'symptoms': 'cough, wheezing', 'diagnosis': '...  
4  {'referral': 'dermatology', 'condition': 'susp...  


**Part 4: Upsert Data into the Index**

In [25]:
index = pc.Index(index_name)
index.upsert_from_dataframe(df)

sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

{'upserted_count': 100}

In [26]:
def is_ready(idx):
   stats = idx.describe_index_stats()
   return stats.total_vector_count > 0

while not is_ready(index):
   time.sleep(5)
print(index.describe_index_stats())

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}


**Part 5: Query & Embedding Function**


In [27]:
from sentence_transformers import SentenceTransformer

# Charger le modèle une fois, pour éviter de le recharger à chaque appel
model = SentenceTransformer("all-MiniLM-L6-v2")

def get_embedding(text):
    return model.encode(text)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Explications

SentenceTransformer("all-MiniLM-L6-v2") est un modèle léger, rapide, et performant pour produire des embeddings textuels de taille 384.

Je place le chargement du modèle en dehors de la fonction pour ne pas le recréer à chaque appel, ce qui améliore les performances.

La fonction get_embedding(text) encode le texte et renvoie un vecteur numérique utilisable pour Pinecone.

In [31]:
question = "What if my patient has leg pain?"
top_k = 5

emb = get_embedding(question)
emb_list = emb.tolist()  # <-- conversion en liste

results = index.query(
    vector=emb_list,
    top_k=top_k,
    include_metadata=True
)

matches = sorted(results.matches, key=lambda m: m.score, reverse=True)

for i, match in enumerate(matches, start=1):
    print(f"{i}. Score: {match.score:.4f}")
    print(f"   Note: {match.metadata.get('text', 'No text metadata available')}\n")



1. Score: 0.5330
   Note: No text metadata available

2. Score: 0.5082
   Note: No text metadata available

3. Score: 0.5082
   Note: No text metadata available

4. Score: 0.4544
   Note: No text metadata available

5. Score: 0.4474
   Note: No text metadata available



**Part 6: Display & Rerank Clinical Notes**

In [57]:
def show_results(q, matches):
    print(f"Question: {q}\n")
    for i, m in enumerate(matches):
        print(f"{i+1}. ID: {m.id}")
        print(f"   Score: {m.score:.4f}")
        print(f"   Métadonnées complètes: {m.metadata}\n")
show_results(question, matches)

Question: What if my patient has leg pain?

1. ID: P0100
   Score: 0.5330
   Métadonnées complètes: {'advice': 'over-the-counter pain relief, stretching', 'symptoms': 'muscle pain'}

2. ID: P095
   Score: 0.5082
   Métadonnées complètes: {'symptoms': 'back pain', 'treatment': 'physical therapy'}

3. ID: P047
   Score: 0.5082
   Métadonnées complètes: {'symptoms': 'back pain', 'treatment': 'physical therapy'}

4. ID: P007
   Score: 0.4544
   Métadonnées complètes: {'surgery': 'knee arthroscopy', 'symptoms': 'pain, swelling', 'treatment': 'physical therapy'}

5. ID: P092
   Score: 0.4474
   Métadonnées complètes: {'condition': 'dehydration', 'treatment': 'IV fluids'}



In [58]:
rerank_docs = [
   {
       "id": m.id,
       "reranking_field": "; ".join([f"{k}: {v}" for k, v in m.metadata.items()])
   }
   for m in matches
]

rerank_query = "What are the best treatment options for leg pain?"  # Exemple de question plus précise


In [59]:
reranked = pc.inference.rerank(
   model="bge-reranker-v2-m3",
   query=rerank_query,
   documents=rerank_docs,
   rank_fields=["reranking_field"],
   top_n=3  # number of top reranked notes to view
)

In [60]:
def show_reranked(q, matches):
    print(f"🔎 Refined Query: {q}\n")
    for i, m in enumerate(matches, 1):
        doc_id = m.document.get("id", "unknown")
        text = m.document.get("reranking_field", "[Aucun contenu]")
        print(f"{i}. ID: {doc_id}")
        print(f"   Score: {m.score:.4f}")
        print(f"   Texte: {text}\n")


In [63]:
def show_reranked(q, matches):
   print(f"Refined Query: {q}")
   for i, m in enumerate(matches):
       print (f"response {i+1}, {m.document.id}, {m.score}, {m.document.reranking_field}")
show_reranked(rerank_query, reranked.data)

Refined Query: What are the best treatment options for leg pain?
response 1, P007, 0.028274601, surgery: knee arthroscopy; symptoms: pain, swelling; treatment: physical therapy
response 2, P0100, 0.005448068, advice: over-the-counter pain relief, stretching; symptoms: muscle pain
response 3, P095, 0.0035936027, symptoms: back pain; treatment: physical therapy
